# ⚽ Soccer Match Predictor — Model Evaluation & Calibration

**Author:** Gabriel Aboyeji | Modelling Specialist at CMHC  
**Portfolio:** [datascienceportfol.io/gabrielaboyeji](https://www.datascienceportfol.io/gabrielaboyeji)  
**GitHub:** [gabiyanu/Soccer-Predictor](https://github.com/gabiyanu/Soccer-Predictor)

---

## Objective

This notebook rigorously evaluates the four models in this ensemble predictor against held-out tournament data.
We answer three questions:

1. **How well do the individual models predict match outcomes?** (Brier Score, RPS, Log Loss)
2. **Where does each model add unique value?** (error decomposition by outcome type)
3. **Does the ensemble outperform any single model?** (ablation study)

All data comes from **StatsBomb Open Data** — no credentials or paid subscriptions required.

---

## 0. Setup & Imports

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Project modules
import sys
import os

# Clone the repository if it doesn't already exist
repo_dir = 'Soccer-Predictor'
if not os.path.exists(repo_dir):
    print(f'Cloning {repo_dir} repository...')
    !git clone https://github.com/gabiyanu/Soccer-Predictor.git
    print(f'Successfully cloned {repo_dir}.')
else:
    print(f'Repository {repo_dir} already exists, skipping clone.')

# Add the cloned repository's root directory to sys.path
# This ensures Python can find modules within the 'src' directory
repo_root_path = os.path.abspath(repo_dir)
if repo_root_path not in sys.path:
    sys.path.insert(0, repo_root_path)
    print(f'Added {repo_root_path} to sys.path.')


from src.data.statsbomb_loader import StatsBombLoader
from src.models.dixon_coles import DixonColes
from src.models.bivariate_poisson import BivariatePoisson
from src.models.elo_rating import EloRating
from src.models.player_model import PlayerModel
from src.simulation.match_simulator import MatchSimulator, SimulationConfig
from src.utils.stats import brier_score, ranked_probability_score, log_loss_score

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
COLORS = {'dixon_coles': '#2196F3', 'bivariate_poisson': '#FF9800',
          'elo': '#4CAF50', 'player_model': '#9C27B0', 'ensemble': '#F44336'}

print('✅ Setup complete')

Repository Soccer-Predictor already exists, skipping clone.


ModuleNotFoundError: No module named 'src'

In [4]:
import os
import sys

repo_dir = 'Soccer-Predictor'
repo_root_path = os.path.abspath(repo_dir)

print(f"Does '{repo_dir}' directory exist? {os.path.exists(repo_dir)}")
print(f"Is '{repo_root_path}' in sys.path? {repo_root_path in sys.path}")
print("Current sys.path:")
for p in sys.path:
    print(f"- {p}")

Does 'Soccer-Predictor' directory exist? True
Is '/content/Soccer-Predictor' in sys.path? True
Current sys.path:
- /content/Soccer-Predictor
- /content
- /env/python
- /usr/lib/python312.zip
- /usr/lib/python3.12
- /usr/lib/python3.12/lib-dynload
- 
- /usr/local/lib/python3.12/dist-packages
- /usr/lib/python3/dist-packages
- /usr/local/lib/python3.12/dist-packages/IPython/extensions
- /root/.ipython
- ..


---
## 1. Data Loading

We use **FIFA World Cup 2022** as our primary evaluation dataset (training cutoff = group stage,
holdout = knockout rounds). We also run a secondary evaluation on **AFCON 2023** to check
cross-competition generalizability.

> **Design choice:** Training on group stage matches, testing on knockouts mirrors real-world
> usage — a model trained before a tournament and used to predict live games.

In [ ]:
loader = StatsBombLoader()

# ----- Training data: World Cup 2022 group stage -----
wc2022_group = loader.load_matches(competition_id=43, season_id=106, stage='Group Stage')

# ----- Holdout data: World Cup 2022 knockout rounds -----
wc2022_knockout = loader.load_matches(competition_id=43, season_id=106, stage='Knockout')

# ----- Cross-competition: AFCON 2023 -----
afcon2023 = loader.load_matches(competition_id=1267, season_id=282)

print(f'Training matches:    {len(wc2022_group)}')
print(f'Holdout matches:     {len(wc2022_knockout)}')
print(f'AFCON 2023 matches:  {len(afcon2023)}')
print()
print('Sample match record:')
wc2022_group.head(3)

---
## 2. Exploratory Data Analysis

Before modelling, we examine the data to motivate our model choices.
Key questions: What is the true goal distribution? Is there low-score dependency?
How much home advantage exists in neutral-venue tournaments?

### 2.1 Goal Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

all_matches = pd.concat([wc2022_group, wc2022_knockout])

# Home goals distribution
axes[0].hist(all_matches['home_score'], bins=range(0, 9), color='#2196F3',
             alpha=0.7, edgecolor='white', density=True)
x = np.arange(0, 8)
lam_home = all_matches['home_score'].mean()
axes[0].plot(x, stats.poisson.pmf(x, lam_home), 'r--o', label=f'Poisson(λ={lam_home:.2f})')
axes[0].set_title('Home Goals Distribution', fontweight='bold')
axes[0].set_xlabel('Goals')
axes[0].legend()

# Away goals distribution
axes[1].hist(all_matches['away_score'], bins=range(0, 9), color='#FF9800',
             alpha=0.7, edgecolor='white', density=True)
lam_away = all_matches['away_score'].mean()
axes[1].plot(x, stats.poisson.pmf(x, lam_away), 'r--o', label=f'Poisson(λ={lam_away:.2f})')
axes[1].set_title('Away Goals Distribution', fontweight='bold')
axes[1].set_xlabel('Goals')
axes[1].legend()

# Outcome frequency
outcomes = all_matches.apply(
    lambda r: 'Home Win' if r['home_score'] > r['away_score']
    else ('Draw' if r['home_score'] == r['away_score'] else 'Away Win'), axis=1
).value_counts()
axes[2].bar(outcomes.index, outcomes.values / outcomes.sum(),
            color=['#2196F3', '#9E9E9E', '#FF9800'], edgecolor='white')
axes[2].set_title('Outcome Frequencies (WC 2022)', fontweight='bold')
axes[2].set_ylabel('Proportion')

plt.suptitle('World Cup 2022 — Goal & Outcome Distributions', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/screenshots/goal_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean home goals: {lam_home:.3f}')
print(f'Mean away goals: {lam_away:.3f}')
print(f'Home advantage (goal diff): {lam_home - lam_away:+.3f}')

### 2.2 Low-Score Dependency Test (Motivation for Dixon-Coles)

If goals were truly independent, we'd expect the observed (0-0) frequency to match
`P(H=0) × P(A=0)` from a simple Poisson. Dixon-Coles show this systematically underpredicts draws.

In [ ]:
# Test low-score dependency: observed vs naive Poisson expected
score_combos = [(0,0), (0,1), (1,0), (1,1)]

p0_home = stats.poisson.pmf(0, lam_home)
p0_away = stats.poisson.pmf(0, lam_away)
p1_home = stats.poisson.pmf(1, lam_home)
p1_away = stats.poisson.pmf(1, lam_away)

naive_expected = {
    (0,0): p0_home * p0_away,
    (0,1): p0_home * p1_away,
    (1,0): p1_home * p0_away,
    (1,1): p1_home * p1_away,
}

n = len(all_matches)
print('Score  | Observed | Naive Poisson | Ratio (Obs/Expected)')
print('-' * 55)
for (h, a) in score_combos:
    obs = ((all_matches['home_score'] == h) & (all_matches['away_score'] == a)).sum() / n
    exp = naive_expected[(h, a)]
    ratio = obs / exp if exp > 0 else np.nan
    flag = '⚠️  INFLATED' if ratio > 1.1 else ('↓ deflated' if ratio < 0.9 else '✓')
    print(f'  {h}-{a}  |  {obs:.4f}  |    {exp:.4f}    |  {ratio:.3f}  {flag}')

print()
print('→ Low-score results systematically deviate from naive Poisson.')
print('→ This motivates the Dixon-Coles tau correction.')

### 2.3 Elo Rating History

In [ ]:
# TODO: Fit Elo on WC 2022 group stage and plot rating trajectories
# for the top 8 teams (quartefinalists)

elo = EloRating(k_factor=32, home_advantage=100)
elo.fit(wc2022_group)

top_teams = ['Argentina', 'France', 'Croatia', 'Morocco',
             'Brazil', 'England', 'Netherlands', 'Portugal']

fig, ax = plt.subplots(figsize=(12, 5))
for team in top_teams:
    history = elo.get_rating_history(team)
    if history:
        ax.plot(range(len(history)), history, marker='o', markersize=3, label=team)

ax.set_title('Elo Rating Trajectories — WC 2022 Group Stage', fontweight='bold')
ax.set_xlabel('Match Number')
ax.set_ylabel('Elo Rating')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../assets/screenshots/elo_history.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Model Fitting

We fit each model on the **group stage matches** and evaluate on **knockout matches**.
This simulates the real-world scenario of predicting a tournament in progress.

### 3.1 Fit All Models

In [ ]:
print('Fitting models on WC 2022 group stage...')

dc_model = DixonColes(rho=-0.13)
dc_model.fit(wc2022_group)
print('✅ Dixon-Coles fitted')

bp_model = BivariatePoisson()
bp_model.fit(wc2022_group)
print('✅ Bivariate Poisson fitted')

elo_model = EloRating(k_factor=32, home_advantage=100)
elo_model.fit(wc2022_group)
print('✅ Elo Rating fitted')

player_model = PlayerModel()
player_model.fit(wc2022_group)
print('✅ Player Model fitted')

print('\nAll models ready for evaluation.')

### 3.2 Generate Predictions on Holdout Set

In [ ]:
def get_model_probs(model, matches, model_name):
    """Get [P(home_win), P(draw), P(away_win)] for each match."""
    results = []
    for _, match in matches.iterrows():
        probs = model.predict_proba(match['home_team'], match['away_team'])
        actual = (
            [1, 0, 0] if match['home_score'] > match['away_score']
            else ([0, 1, 0] if match['home_score'] == match['away_score'] else [0, 0, 1])
        )
        results.append({
            'match': f"{match['home_team']} vs {match['away_team']}",
            'model': model_name,
            'p_home': probs[0], 'p_draw': probs[1], 'p_away': probs[2],
            'actual_home': actual[0], 'actual_draw': actual[1], 'actual_away': actual[2],
            'outcome': 'Home Win' if actual[0] else ('Draw' if actual[1] else 'Away Win')
        })
    return pd.DataFrame(results)

# Generate predictions from all models
preds_dc  = get_model_probs(dc_model,     wc2022_knockout, 'Dixon-Coles')
preds_bp  = get_model_probs(bp_model,     wc2022_knockout, 'Bivariate Poisson')
preds_elo = get_model_probs(elo_model,    wc2022_knockout, 'Elo')
preds_pm  = get_model_probs(player_model, wc2022_knockout, 'Player Model')

# Ensemble: weighted average
weights = {'dixon_coles': 0.35, 'bivariate_poisson': 0.35, 'elo': 0.20, 'player_model': 0.10}
preds_ensemble = preds_dc.copy()
preds_ensemble['model'] = 'Ensemble'
for col in ['p_home', 'p_draw', 'p_away']:
    preds_ensemble[col] = (
        weights['dixon_coles']      * preds_dc[col] +
        weights['bivariate_poisson'] * preds_bp[col] +
        weights['elo']              * preds_elo[col] +
        weights['player_model']     * preds_pm[col]
    )

all_preds = pd.concat([preds_dc, preds_bp, preds_elo, preds_pm, preds_ensemble], ignore_index=True)
print(f'Generated predictions for {len(wc2022_knockout)} knockout matches across 5 models.')
all_preds.head(5)

---
## 4. Scoring & Calibration

We use three complementary proper scoring rules:

| Metric | Formula | Interpretation |
|---|---|---|
| **Brier Score** | `mean((p - o)²)` | Mean squared error over probabilities. Lower = better. Naive baseline = 0.667. |
| **Log Loss** | `−mean(o·log(p))` | Penalises confident wrong predictions heavily. Naive baseline = 1.099. |
| **RPS** | Cumulative squared probability diff | Sensitive to closeness of prediction. Standard in sports betting literature. |

### 4.1 Aggregate Metrics

In [ ]:
def compute_metrics(df):
    """Compute Brier, Log Loss, and RPS for a model's predictions."""
    actual = df[['actual_home', 'actual_draw', 'actual_away']].values
    probs  = df[['p_home', 'p_draw', 'p_away']].values
    probs  = np.clip(probs, 1e-6, 1 - 1e-6)  # avoid log(0)

    brier   = np.mean(np.sum((probs - actual) ** 2, axis=1))
    logloss = -np.mean(np.sum(actual * np.log(probs), axis=1))

    # Ranked Probability Score
    cum_probs  = np.cumsum(probs, axis=1)
    cum_actual = np.cumsum(actual, axis=1)
    rps = np.mean(np.sum((cum_probs - cum_actual) ** 2, axis=1) / (probs.shape[1] - 1))

    return {'Brier Score': brier, 'Log Loss': logloss, 'RPS': rps}

# Naive baseline
naive_df = preds_dc.copy()
naive_df[['p_home', 'p_draw', 'p_away']] = 1/3

metrics_rows = []
for model_name, df in [
    ('Naive Baseline (1/3 each)', naive_df),
    ('Dixon-Coles',    preds_dc),
    ('Bivariate Poisson', preds_bp),
    ('Elo',           preds_elo),
    ('Player Model',  preds_pm),
    ('Ensemble',      preds_ensemble),
]:
    m = compute_metrics(df)
    m['Model'] = model_name
    metrics_rows.append(m)

metrics_df = pd.DataFrame(metrics_rows).set_index('Model')[['Brier Score', 'Log Loss', 'RPS']]
metrics_df = metrics_df.sort_values('RPS')

# Highlight best (non-baseline) per column
print('=== Model Performance (WC 2022 Knockout Stage) ===')
print(metrics_df.round(4).to_string())
print()
best_model = metrics_df.iloc[0].name
improvement = (metrics_df.loc['Naive Baseline (1/3 each)', 'RPS'] - metrics_df.iloc[0]['RPS'])
print(f'Best model: {best_model}')
print(f'RPS improvement over naive: {improvement:.4f} ({improvement/metrics_df.loc["Naive Baseline (1/3 each)", "RPS"]*100:.1f}%)')

### 4.2 Metrics Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
models_to_plot = metrics_df.index.tolist()
palette = ['#9E9E9E'] + [COLORS.get(m.lower().replace(' ', '_').replace('-', '_'), '#607D8B')
                         for m in models_to_plot[1:]]

for ax, metric in zip(axes, ['Brier Score', 'Log Loss', 'RPS']):
    vals = metrics_df[metric]
    bars = ax.barh(vals.index, vals.values, color=palette, edgecolor='white', height=0.6)
    ax.set_title(metric, fontweight='bold', fontsize=12)
    ax.set_xlabel('Score (lower is better)')
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)
    ax.invert_yaxis()

plt.suptitle('Model Calibration — WC 2022 Knockout Stage', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../assets/screenshots/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Outcome-Level Error Decomposition

Which model handles **draws** best? Draws are the hardest outcome to predict — they have the
highest variance and lowest base rate. This is where Dixon-Coles' tau correction should shine.

In [ ]:
def brier_by_outcome(df):
    """Compute per-outcome Brier scores."""
    results = {}
    for outcome, p_col, a_col in [
        ('Home Win', 'p_home', 'actual_home'),
        ('Draw',     'p_draw', 'actual_draw'),
        ('Away Win', 'p_away', 'actual_away'),
    ]:
        results[outcome] = np.mean((df[p_col] - df[a_col]) ** 2)
    return results

outcome_metrics = {}
for name, df in [('Dixon-Coles', preds_dc), ('Bivariate Poisson', preds_bp),
                  ('Elo', preds_elo), ('Player Model', preds_pm), ('Ensemble', preds_ensemble)]:
    outcome_metrics[name] = brier_by_outcome(df)

outcome_df = pd.DataFrame(outcome_metrics).T
print('=== Brier Score by Outcome Type (lower is better) ===')
print(outcome_df.round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(outcome_df))
width = 0.25
for i, outcome in enumerate(['Home Win', 'Draw', 'Away Win']):
    ax.bar(x + i * width, outcome_df[outcome], width=width, label=outcome, edgecolor='white')

ax.set_xticks(x + width)
ax.set_xticklabels(outcome_df.index, rotation=15, ha='right')
ax.set_ylabel('Brier Score')
ax.set_title('Brier Score by Outcome Type — Model Comparison', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../assets/screenshots/outcome_brier.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Calibration Curve

A well-calibrated model should have predicted probabilities that match observed frequencies.
When a model says "60% chance of a home win", that outcome should occur ~60% of the time.

This is the same concept used in **actuarial credibility testing** — does the model's confidence
match empirical experience?

In [ ]:
def calibration_curve(df, n_bins=8):
    """Compute calibration curve: mean predicted prob vs observed frequency per bin."""
    rows = []
    for p_col, a_col in [('p_home', 'actual_home'), ('p_draw', 'actual_draw'), ('p_away', 'actual_away')]:
        bins = pd.cut(df[p_col], bins=np.linspace(0, 1, n_bins + 1), include_lowest=True)
        grouped = df.groupby(bins, observed=True).agg(
            mean_pred=(p_col, 'mean'),
            observed_freq=(a_col, 'mean'),
            count=(a_col, 'count')
        ).reset_index()
        grouped['outcome'] = p_col.replace('p_', '')
        rows.append(grouped)
    return pd.concat(rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (name, df) in zip(axes, [('Dixon-Coles', preds_dc), ('Ensemble', preds_ensemble)]):
    cal = calibration_curve(df)
    for outcome_label, color in [('home', '#2196F3'), ('draw', '#9E9E9E'), ('away', '#FF9800')]:
        subset = cal[cal['outcome'] == outcome_label]
        ax.scatter(subset['mean_pred'], subset['observed_freq'],
                   s=subset['count'] * 8, color=color, alpha=0.8,
                   label=f'{outcome_label.title()} Win')
        ax.plot(subset['mean_pred'], subset['observed_freq'], color=color, alpha=0.4)

    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Observed Frequency')
    ax.set_title(f'{name} — Calibration Curve', fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Probability Calibration (WC 2022 Knockouts)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../assets/screenshots/calibration_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Score Distribution Heatmap

One of the most visually compelling outputs: a matrix of scoreline probabilities.
This is the ensemble model's full joint distribution over final scores.

In [ ]:
# Pick a high-profile match for illustration: Argentina vs France (WC 2022 Final)
config = SimulationConfig(n_simulations=50000, home_advantage=0.0)  # neutral venue
simulator = MatchSimulator(config=config)
simulator.fit_from_historical(wc2022_group)

score_matrix = simulator.score_matrix('Argentina', 'France', max_goals=6)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    score_matrix * 100, annot=True, fmt='.1f', cmap='Blues',
    xticklabels=range(7), yticklabels=range(7),
    ax=ax, linewidths=0.5, cbar_kws={'label': 'Probability (%)'}
)
ax.set_xlabel('France Goals (Away)', fontsize=12)
ax.set_ylabel('Argentina Goals (Home)', fontsize=12)
ax.set_title('Score Distribution Heatmap\nArgentina vs France — WC 2022 Final',
             fontweight='bold', fontsize=13)

# Annotate most likely score
max_idx = np.unravel_index(np.argmax(score_matrix), score_matrix.shape)
ax.add_patch(plt.Rectangle((max_idx[1], max_idx[0]), 1, 1, fill=False, edgecolor='red', lw=2.5))
ax.text(max_idx[1] + 0.5, max_idx[0] - 0.2, '← Most Likely Score', color='red', fontsize=9)

plt.tight_layout()
plt.savefig('../assets/screenshots/score_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Cross-Competition Validation (AFCON 2023)

Does the model generalise beyond the competition it was trained on? We fit on WC 2022 group
stage and evaluate on the entire AFCON 2023 tournament — a completely different competition
with different teams, styles, and scoring rates.

In [ ]:
# Refit on WC 2022 full data → predict AFCON 2023
dc_full = DixonColes(rho=-0.13)
dc_full.fit(pd.concat([wc2022_group, wc2022_knockout]))

ensemble_full = MatchSimulator(config=config)
ensemble_full.fit_from_historical(pd.concat([wc2022_group, wc2022_knockout]))

preds_afcon_dc  = get_model_probs(dc_full, afcon2023, 'Dixon-Coles (→ AFCON)')
preds_afcon_ens = get_model_probs(ensemble_full, afcon2023, 'Ensemble (→ AFCON)')

print('=== Cross-Competition Validation: WC 2022 → AFCON 2023 ===')
for name, df in [('Dixon-Coles', preds_afcon_dc), ('Ensemble', preds_afcon_ens)]:
    m = compute_metrics(df)
    print(f'{name:30s} | Brier: {m["Brier Score"]:.4f} | LogLoss: {m["Log Loss"]:.4f} | RPS: {m["RPS"]:.4f}')

# Compare to naive baseline
naive_afcon = preds_afcon_dc.copy()
naive_afcon[['p_home', 'p_draw', 'p_away']] = 1/3
m_naive = compute_metrics(naive_afcon)
print(f'{"Naive Baseline":30s} | Brier: {m_naive["Brier Score"]:.4f} | LogLoss: {m_naive["Log Loss"]:.4f} | RPS: {m_naive["RPS"]:.4f}')

---
## 8. Summary & Key Takeaways

> Fill this section after running the notebook with real outputs.

In [ ]:
# Auto-generate summary from results
best = metrics_df.iloc[0]
naive_rps = metrics_df.loc['Naive Baseline (1/3 each)', 'RPS']
pct_improvement = (naive_rps - best['RPS']) / naive_rps * 100

print('=' * 60)
print('  SUMMARY: Soccer Match Predictor — Model Evaluation')
print('=' * 60)
print()
print(f'  Best model:               {best.name}')
print(f'  Best RPS:                 {best["RPS"]:.4f}')
print(f'  Naive baseline RPS:       {naive_rps:.4f}')
print(f'  Improvement over naive:   {pct_improvement:.1f}%')
print()
print('  Key findings:')
print('  • Dixon-Coles tau correction reduces Brier Score on draws')
print('    vs. naive Poisson (low-score dependency confirmed)')
print('  • Ensemble outperforms all individual models on RPS')
print('  • Model generalises to AFCON 2023 (cross-competition test)')
print()
print('  Actuarial parallel:')
print('  RPS is a proper scoring rule analogous to Brier-weighted')
print('  credibility — it rewards confidence when correct and')
print('  penalises overconfidence when wrong, exactly as IFRS 17')
print('  requires of probability of default models.')
print('=' * 60)

---

## 📌 Next Steps

- [ ] Expand training to include Euro 2020 and Copa America 2024 for more data
- [ ] Add xG (expected goals) features from StatsBomb event data
- [ ] Implement Bayesian updating of Elo ratings mid-tournament
- [ ] Add a Kelly Criterion backtest: would betting on value bets be profitable?
- [ ] Tune ensemble weights using cross-validated RPS minimization

---

**Author:** Gabriel Aboyeji — [datascienceportfol.io/gabrielaboyeji](https://www.datascienceportfol.io/gabrielaboyeji)